# Indian E-commerce Returns & Fulfilment Analysis
## Exploratory Data Analysis (EDA)

**Goal:** Identify useful patterns in item-level sales, returns, refunds, customers, and delivery performance.

**Data grain:** One row per order item. Use `item_total` for sales and distinct `order_id` for order counts. Recorded refund amounts do not confirm that refunds were settled.

## 1. Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("cleaned_ecommerce_order_items.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Run Data_Cleaning_and_Preparation.ipynb first.")

df = pd.read_csv(DATA_PATH, parse_dates=["order_datetime", "shipped_datetime", "delivered_datetime"])
sns.set_theme(style="whitegrid")
print("Dataset shape:", df.shape)
display(df.head())


## 2. Quick Data Checks

In [ ]:
checks = pd.Series({
    "Unique orders": df["order_id"].nunique(),
    "Unique customers": df["customer_id"].nunique(),
    "Duplicate order-item keys": df.duplicated(["order_id", "order_item_id"]).sum(),
    "Missing order IDs": df["order_id"].isna().sum()
})
display(checks.to_frame("value"))


## 3. Key Business KPIs

`item_total` is item-level sales. Return and delivery rates use records with known flags only.

In [ ]:
known_returns = df["is_returned"].notna()
known_delivery = df["is_late_delivery"].notna()

kpis = pd.Series({
    "Order-item rows": len(df),
    "Unique orders": df["order_id"].nunique(),
    "Unique customers": df["customer_id"].nunique(),
    "Units sold": df["quantity"].sum(),
    "Item-level sales": df["item_total"].sum(),
    "Returned items": df["is_returned"].eq(1).sum(),
    "Returned-item rate (%)": df.loc[known_returns, "is_returned"].mean() * 100,
    "Late-delivery rate (%)": df.loc[known_delivery, "is_late_delivery"].mean() * 100,
    "Average order value": df.groupby("order_id")["item_total"].sum().mean()
})
display(kpis.to_frame("value").round(2))


## 4. Univariate Analysis

In [ ]:
# Order status distribution
status_counts = df["order_status"].value_counts(dropna=False)
display(status_counts.to_frame("order_item_rows"))
status_counts.plot(kind="bar", figsize=(8, 4), title="Order Status Distribution")
plt.xlabel("Order status"); plt.ylabel("Order-item rows"); plt.xticks(rotation=35)
plt.tight_layout(); plt.show()

# Item-level sales distribution
df["item_total"].dropna().plot(kind="hist", bins=30, figsize=(7, 4), title="Item Sales Distribution")
plt.xlabel("Item total"); plt.ylabel("Frequency"); plt.tight_layout(); plt.show()


## 5. Bivariate Analysis

In [ ]:
# Sales by category
category_sales = df.groupby("category", dropna=False)["item_total"].sum().sort_values()
display(category_sales.sort_values(ascending=False).to_frame("item_sales"))
category_sales.plot(kind="barh", figsize=(8, 5), title="Item-level Sales by Category")
plt.xlabel("Item-level sales"); plt.ylabel("Category"); plt.tight_layout(); plt.show()

# Return rate by category; unknown flags are excluded from the denominator
category_returns = df.groupby("category", dropna=False).agg(
    known_flags=("is_returned", "count"), returned_items=("is_returned", lambda x: x.eq(1).sum())
)
category_returns["return_rate_pct"] = 100 * category_returns["returned_items"] / category_returns["known_flags"]
display(category_returns.sort_values("return_rate_pct", ascending=False).round(2))
category_returns["return_rate_pct"].sort_values().plot(kind="barh", figsize=(8, 5), title="Returned-item Rate by Category")
plt.xlabel("Return rate (%)"); plt.ylabel("Category"); plt.tight_layout(); plt.show()

# Most common return reasons
reasons = df.loc[df["is_returned"].eq(1), "return_reason"].dropna().value_counts()
display(reasons.to_frame("returned_items"))
reasons.sort_values().plot(kind="barh", figsize=(8, 4), title="Return Reasons")
plt.xlabel("Returned items"); plt.ylabel("Return reason"); plt.tight_layout(); plt.show()


## 6. Multivariate Analysis

In [ ]:
# Compare return rates by category and delivery status
valid = df.dropna(subset=["category", "is_late_delivery", "is_returned"]).copy()
valid["delivery_status"] = valid["is_late_delivery"].map({0: "Not Late", 1: "Late"})
return_matrix = valid.pivot_table(
    index="category", columns="delivery_status", values="is_returned", aggfunc="mean"
) * 100

display(return_matrix.round(2))
sns.heatmap(return_matrix, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Return Rate by Category and Delivery Status (%)")
plt.xlabel("Delivery status"); plt.ylabel("Category"); plt.tight_layout(); plt.show()


## 7. Correlation Analysis

In [ ]:
numeric_cols = [
    "quantity", "mrp", "discount_pct", "selling_price", "item_total",
    "promised_delivery_days", "actual_delivery_days", "seller_rating"
]
correlation = df[numeric_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Between Selected Numerical Variables")
plt.tight_layout(); plt.show()
print("Correlation shows association, not causation.")


## 8. Outlier Detection

In [ ]:
outlier_cols = ["item_total", "selling_price", "actual_delivery_days"]

for col in outlier_cols:
    values = df[col].dropna()
    plt.figure(figsize=(7, 2.5))
    sns.boxplot(x=values)
    plt.title(f"Outlier Check: {col}")
    plt.tight_layout(); plt.show()

    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((values < lower) | (values > upper)).sum()
    print(f"{col}: {count:,} potential outliers (review; do not delete automatically)")


## 9. Delivery Performance

In [ ]:
delivery = df.groupby("courier_partner", dropna=False).agg(
    items=("order_item_id", "size"),
    late_items=("is_late_delivery", lambda x: x.eq(1).sum()),
    known_flags=("is_late_delivery", "count"),
    avg_delivery_days=("actual_delivery_days", "mean")
)
delivery["late_rate_pct"] = 100 * delivery["late_items"] / delivery["known_flags"]
display(delivery.sort_values("late_rate_pct", ascending=False).round(2))
delivery["late_rate_pct"].sort_values().plot(kind="barh", figsize=(8, 4), title="Late-delivery Rate by Courier")
plt.xlabel("Late-delivery rate (%)"); plt.ylabel("Courier partner"); plt.tight_layout(); plt.show()


## 10. Monthly Trend and Customer/Payment Analysis

In [ ]:
monthly = df.dropna(subset=["order_datetime"]).copy()
monthly["order_month"] = monthly["order_datetime"].dt.to_period("M").dt.to_timestamp()
monthly_sales = monthly.groupby("order_month").agg(
    orders=("order_id", "nunique"), item_sales=("item_total", "sum"),
    returned_items=("is_returned", lambda x: x.eq(1).sum())
)
display(monthly_sales.round(2))
monthly_sales["item_sales"].plot(figsize=(9, 4), marker="o", title="Monthly Item-level Sales")
plt.xlabel("Month"); plt.ylabel("Item-level sales"); plt.tight_layout(); plt.show()

print("Sales and return rate by Plus membership")
member_summary = df.groupby("is_plus_member").agg(
    orders=("order_id", "nunique"), item_sales=("item_total", "sum"),
    return_rate_pct=("is_returned", lambda x: 100 * x.mean())
)
display(member_summary.round(2))

print("Sales by payment method")
display(df.groupby("payment_method").agg(
    orders=("order_id", "nunique"), item_sales=("item_total", "sum")
).sort_values("item_sales", ascending=False).round(2))

print("Sales and return rate by region")
region_summary = df.groupby("region").agg(
    orders=("order_id", "nunique"), item_sales=("item_total", "sum"),
    return_rate_pct=("is_returned", lambda x: 100 * x.mean())
)
display(region_summary.sort_values("item_sales", ascending=False).round(2))


## 11. Refund Analysis

In [ ]:
refunds = df.loc[df["is_returned"].eq(1) & df["refund_amount"].notna()]
refund_summary = refunds.groupby("refund_mode").agg(
    records=("refund_amount", "count"), recorded_refund_amount=("refund_amount", "sum"),
    average_recorded_refund=("refund_amount", "mean")
).sort_values("recorded_refund_amount", ascending=False)
display(refund_summary.round(2))
print("Recorded refund amounts do not confirm that refunds were settled.")


## 12. Summary

Use the results above to report the strongest findings on category sales, returns, delivery delays, and monthly trends. Recommend actions only when supported by the results. Do not treat correlation as causation or recorded refunds as confirmed payments.